Environment: Harpy

### Import packages

In [3]:
import anndata as ad
import harpy as hp
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from skimage import io, morphology
from matplotlib import pyplot as plt
import gc 

### Specify paths

In [18]:
df = pd.read_csv(r'e:\Nova-ST\Exp65-sampleSPC014-novascope_30Jun2025\novascope_output\analysis\Exp65-sampleSPC014\Exp65-sampleSPC014-default\preprocess\Exp65-sampleSPC014-default.feature.tsv.gz', delimiter='\t')

In [21]:
path_hne_image = r'e:\Nova-ST\Exp65-sampleSPC014-novascope_30Jun2025\input\HE\Exp108-NovaST-mousebrain_10X.tif'
path_transcripts = r'e:\Nova-ST\Exp65-sampleSPC014-novascope_30Jun2025\novascope_output\analysis\Exp65-sampleSPC014\Exp65-sampleSPC014-default\preprocess\Exp65-sampleSPC014-default.transcripts.tsv.gz'
output_path = r'E:\Nova-ST\output'
plots_path = os.path.join(output_path, 'plots')
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

In [5]:
import spatialdata as sd
sdata = sd.read_zarr(r'e:\Nova-ST\output\sdata.zarr')

c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04


### Create sdata

In [3]:
hne_img_layer = "hne_image_rgb"

sdata = hp.io.create_sdata(
    input=path_hne_image,
    output_path=os.path.join(output_path, "sdata.zarr"),
    img_layer=hne_img_layer,
    chunks=1024,
    scale_factors=(2, 2, 2, 2),
)
sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
<tifffile.TiffFile 'Exp108-NovaST-mousebrain_10X.tif'> OME series cannot handle discontiguous storage ((14294, 18695, 3) != (3, 14294, 18695))
<tifffile.TiffFile 'Exp108-NovaST-mousebrain_10X.tif'> OME series cannot handle discontiguous storage ((14294, 18695, 3) != (3, 14294, 18695))


INFO     The Zarr backing store has been changed from None the new file path: E:\Nova-ST\output\sdata.zarr         


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-07-18 09:09:18,764 - harpy.image._manager - INFO - Writing results to layer 'hne_image_rgb'
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: E:\Nova-ST\output\sdata.zarr
└── Images
      └── 'hne_image_rgb': DataTree[cyx] (3, 14294, 18695), (3, 7147, 9347), (3, 3573, 4673), (3, 1786, 2336), (3, 893, 1168)
with coordinate systems:
    ▸ 'global', with elements:
        hne_image_rgb (Images)

In [4]:
# Plot H&E
import spatialdata_plot
import xarray
xa: xarray.DataArray = sdata.images[hne_img_layer][f'scale0']['image']
shape = (xa.shape[1], xa.shape[2])

figsize = (shape[1]/800, shape[0]/800)
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("y", fontsize=12)

from matplotlib.colors import Normalize

sdata.pl.render_images(
    element=hne_img_layer
).pl.show(
    coordinate_systems="global",
    title="H&E",
    figsize=figsize,
    dpi=300,
    colorbar=False,
    ax=ax,
)

save_path = os.path.join(output_path, 'plots', f'HE.png')
fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:\Nova-ST\output\plots\HE.png


In [20]:
df

,gene_id,gene,gn,gt,spl,unspl,ambig
0,ENSMUSG00000051951,Xkr4,75,604,74,522,1
1,ENSMUSG00000089699,Gm1992,0,2,0,0,0
2,ENSMUSG00000102331,Gm19938,181,0,0,0,0
3,ENSMUSG00000102343,Gm37381,0,0,0,0,0
4,ENSMUSG00000025900,Rp1,0,11,0,8,0
...,...,...,...,...,...,...,...
32280,ENSMUSG00000095523,AC124606.1,0,0,0,0,0
32281,ENSMUSG00000095475,AC133095.2,0,0,0,0,0
32282,ENSMUSG00000094855,AC133095.1,0,0,0,0,0
32283,ENSMUSG00000095019,AC234645.1,0,0,0,0,0


In [22]:
sdata = hp.io.read_transcripts(
    sdata,
    path_count_matrix=path_transcripts,
    column_x=2,
    column_y=3,
    column_z=None,
    column_gene=5,       # column 5 is the "readable" gene name, column 4 is the Ensemble Gene ID
    column_midcount=6,
    delimiter='\t',
    header=0,
    comment=None,
    transform_matrix=np.array([[0.001, 0, 0],[0, 0.001, 0],[0, 0, 1]]),
    overwrite=True
)
sdata

c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\dask\dataframe\io\csv.py:555: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(
2025-07-18 11:30:48,732 - harpy.io._transcripts - INFO - Transform matrix used:
 [[0.001 0.    0.   ]
 [0.    0.001 0.   ]
 [0.    0.    1.   ]]
2025-07-18 11:36:10,083 - harpy.utils._io - WARNING - layer with name 'transcripts' already exists. Overwriting...


SpatialData object, with associated Zarr store: E:\Nova-ST\output\sdata.zarr
├── Images
│     ├── 'hne_image_rgb': DataTree[cyx] (3, 14294, 18695), (3, 7147, 9347), (3, 3573, 4673), (3, 1786, 2336), (3, 893, 1168)
│     └── 'transcript_density': DataArray[cyx] (1, 14294, 18695)
├── Labels
│     ├── 'labels_spots_10um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_20um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_40um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_100um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     └── 'tissue_mask': DataArray[yx] (11896, 13902)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'shapes_spots_10um': GeoDataFrame shape: (4110990, 1) (2D shapes)
│     ├── 'shapes_spots_20u

### Plot transcript density

In [23]:
sdata = hp.im.transcript_density(
    sdata,
    img_layer='hne_image_rgb',
    points_layer='transcripts',
    output_layer='transcript_density',
    overwrite=True
)
sdata

2025-07-18 11:36:28,662 - harpy.image._transcripts - INFO - The number of transcripts ( 36607421 ) is larger than n_sample, sampling 15000000 transcripts.
2025-07-18 11:36:28,678 - harpy.image._transcripts - INFO - sampling finished
2025-07-18 11:36:28,742 - harpy.image._manager - INFO - Writing results to layer 'transcript_density'
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
2025-07-18 11:36:57,532 - harpy.utils._io - WARNING - layer with name 'transcript_density' already exists. Overwriting...
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: E:\Nova-ST\output\sdata.zarr
├── Images
│     ├── 'hne_image_rgb': DataTree[cyx] (3, 14294, 18695), (3, 7147, 9347), (3, 3573, 4673), (3, 1786, 2336), (3, 893, 1168)
│     └── 'transcript_density': DataArray[cyx] (1, 14294, 18695)
├── Labels
│     ├── 'labels_spots_10um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_20um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_40um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     ├── 'labels_spots_100um': DataTree[yx] (14294, 18695), (7147, 9347), (3573, 4673), (1786, 2336), (893, 1168)
│     └── 'tissue_mask': DataArray[yx] (11896, 13902)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'shapes_spots_10um': GeoDataFrame shape: (4110990, 1) (2D shapes)
│     ├── 'shapes_spots_20u

In [24]:
hp.pl.transcript_density(
    sdata,
    img_layer=[hne_img_layer, "transcript_density"],
    channel=0,
    output=os.path.join(plots_path, "transcript_density.png")
)

### Add hexagonal grids

In [8]:
# Create hexagonal grids
for lvl in [10, 20, 40, 100]:
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape, 
        size=lvl/2,
        output_shapes_layer=f"shapes_spots_{lvl}um", 
        output_labels_layer=f"labels_spots_{lvl}um", 
        grid_type='hexagon', 
        offset=(0, 0), 
        chunks=1024, 
        client=None, 
        transformations=None, 
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

2025-07-18 09:22:31,671 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_10um'
2025-07-18 09:29:10,887 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_20um'
2025-07-18 09:31:07,313 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_40um'
2025-07-18 09:31:51,422 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_100um'


In [9]:
# Plot hexagonal grid labels
for lvl in [10, 20, 40, 100]:  
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    figsize = (shape[1]/150, shape[0]/150)
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)

    sdata.pl.render_images(
        element=hne_img_layer
    ).pl.render_labels(
        element=f"labels_spots_{lvl}um", fill_alpha=0.1, contour_px=5, outline_alpha=1
    ).pl.show(
        coordinate_systems="global",
        figsize=figsize,
        dpi=300,
        colorbar=False,
        ax=ax,
    )

    save_path = os.path.join(plots_path_lvl, f'plot_grid_{lvl}um.png')
    fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

MemoryError: Unable to allocate 18.9 GiB for an array with shape (22012, 28790, 4) and data type float64

Error in callback <function _draw_all_if_interactive at 0x00000214755A9F30> (for post_execute), with arguments args (),kwargs {}:


MemoryError: Unable to allocate 9.44 GiB for an array with shape (22012, 28790, 4) and data type float32

MemoryError: Unable to allocate 9.44 GiB for an array with shape (22012, 28790, 4) and data type float32

<Figure size 37390x28588 with 1 Axes>

In [25]:
import gc
gc.collect()

73897

In [26]:
# Create tables
for lvl in [10, 20, 40, 100]:  
    sdata = hp.tb.allocate(
        sdata=sdata,
        labels_layer=f"labels_spots_{lvl}um",
        points_layer="transcripts",
        output_layer=f"{lvl}um",
        update_shapes_layers=False,
        overwrite=True,
    )

2025-07-18 11:37:55,570 - harpy.table._allocation - INFO - Calculating cell counts.
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_100um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_10um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_20um', which is not present in 

### Add tissue mask

In [27]:
import geopandas as gpd
gdf = gpd.read_file(r'e:\Nova-ST\tissue_annotation\tissue.geojson')
gdf.index = gdf.index + 1

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = 'tissue',
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = 'tissue',
    output_layer = 'tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-07-18 11:43:08,519 - harpy.utils._io - WARNING - layer with name 'tissue' already exists. Overwriting...
2025-07-18 11:43:14,926 - harpy.image._manager - INFO - Writing results to layer 'tissue_mask'
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-07-18 11:43:44,449 - harpy.utils._io - WARNING - layer with name 'tissue_mask' already exists. Overwriting...


In [ ]:
# from napari_spatialdata import Interactive
# Interactive(sdata)

2025-07-18 11:54:14.610 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-07-18 11:54:15.418 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


### Filter out spots not in tissue

In [29]:
def add_in_tissue(sdata, table_layer, tissue_mask):
    def assign_region(centroid, mask, mask_shape):
        x, y = int(round(centroid[0])), int(round(centroid[1]))
        
        if 0 <= y < mask_shape[0] and 0 <= x < mask_shape[1]:
            return 1 if mask[y, x] == 1 else 0
        return 0  # Outside mask bounds

    adata = sdata.tables[table_layer].copy()
    coordinates = adata.obsm['spatial']
    coordinates_df = pd.DataFrame(coordinates, columns=["x", "y"], index=adata.obs.index)

    mask_shape = tissue_mask.shape
    adata.obs["in_tissue"] = coordinates_df.apply(
        lambda row: assign_region((row["x"], row["y"]), tissue_mask, mask_shape), axis=1
    )

    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=table_layer,
        region=[f"labels_spots_{table_layer}"],
        overwrite=True,
    )
    
    adata_in_tissue = adata[adata.obs["in_tissue"] == 1].copy()
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata_in_tissue.copy(),
        output_layer=f'{table_layer}_in_tissue',
        region=[f"labels_spots_{table_layer}"],
        overwrite=True,
    )

    return sdata


In [30]:
for lvl in [10, 20, 40, 100]:
    sdata = add_in_tissue(
        sdata, 
        table_layer = f'{lvl}um', 
        tissue_mask = sdata.labels['tissue_mask'].compute().data
    )

c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_100um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_10um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_20um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\spatialdata\_core\spatialdata.py:184: UserWarning: The table is annotating 'labels_spots_40um', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)

### Leiden clustering

In [ ]:
for lvl in [100]:
    print(f'running: {lvl}um')

    labels_layer=f"labels_spots_{lvl}um"
    shapes_layer=f"shapes_spots_{lvl}um"
    table_layer=f"{lvl}um"
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=table_layer,
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps = 50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.6]

    for res in leiden_resolutions:
        res_str = str(res).replace('.', '')
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=True,
            calculate_neighbors=True,
            rank_genes=True,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(plots_path_lvl, f'UMAP_leiden_{res_str}')
        )
        
        figsize = (shape[1]/400, shape[0]/400)
        fig, ax = plt.subplots(figsize=figsize, dpi=300)
        ax.set_xlabel("x", fontsize=12)
        ax.set_ylabel("y", fontsize=12)

        sdata.pl.render_images(
            element=hne_img_layer
        ).pl.render_labels(
            element=labels_layer,
            color=f'leiden_{res_str}',
            table_name=f"{table_layer}_processed",
            cmap="rainbow",
            fill_alpha=1, 
            contour_px=1, 
            outline_alpha=0
        ).pl.show(
            coordinate_systems="global",
            title=f"Leiden {res_str}",
            figsize=figsize,
            dpi=300,
            colorbar=True,
            ax=ax,
        )

        save_path = os.path.join(plots_path_lvl, f'leiden_{res_str}.png')
        fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)
    
        gc.collect()

running: 100um


2025-07-18 11:53:44,710 - harpy.table._preprocess - INFO - Calculating cell size from provided labels_layer 'labels_spots_100um'


TypeError: normalize_total() got an unexpected keyword argument 'layers'

### Plot QC metrics

In [ ]:
for lvl in [100]:
    labels_layer=f"labels_spots_{lvl}um"
    shapes_layer=f"shapes_spots_{lvl}um"
    table_layer = f'{lvl}um_in_tissue_processed'
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    for col in ['total_counts', 'n_genes_by_counts']:
        figsize = (shape[1]/400, shape[0]/400)
        fig, ax = plt.subplots(figsize=figsize, dpi=300)
        ax.set_xlabel("x", fontsize=12)
        ax.set_ylabel("y", fontsize=12)

        sdata.pl.render_images(
            element=hne_img_layer
        ).pl.render_labels(
            element=labels_layer,
            color=col,
            table_name=table_layer,
            cmap="viridis",
            fill_alpha=1, 
            contour_px=1, 
            outline_alpha=0
        ).pl.show(
            coordinate_systems="global",
            title=col,
            figsize=figsize,
            dpi=300,
            colorbar=True,
            ax=ax,
        )

        save_path = os.path.join(plots_path_lvl, f'{col}_{table_layer}.png')
        fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)
        
        gc.collect()